# Do Bias Metrics Agree?

Benchmarking AIF360 and Fairlearn across fairness metrics and mitigation strategies (AIF360 Reweighing + Fairlearn ExponentiatedGradient).

Datasets: German Credit + COMPAS  
Models: Logistic Regression (baseline) + Random Forest  
Frameworks: AIF360 + Fairlearn  

Course: Data Science & Artificial Intelligence 2
Instructor: Prof. Sabrina Kirrane 

Victor Gabriel Dinu, Eddie Begovic / Summer Semester 2026  

---
## Background and Introduction

This section is here so the comparison that follows makes sense even to someone who comes across these libraries for the first time. AIF360 and Fairlearn are both free, open-source Python libraries for measuring and reducing unfair treatment of protected groups by a machine learning model. They were both released around 2018, they both sit on top of scikit-learn style models, and in essence they measure the same things. The interesting part for this project is where they differ, because those differences are what this whole notebook is built to explore.

### AIF360 (AI Fairness 360)

AIF360 comes from IBM Research. One can think of it as a large, reference toolbox: it comes packed with around 70 fairness metrics and around 10 mitigation algorithms, spread across all three stages of the pipeline (pre-processing, in-processing and post-processing). If the idea of fairness has shown up in a paper in this domain, there is a good chance AIF360 has somehow been implemented (Bellamy et al., 2019).

The slight downside of its wide applicability is that AIF360 needs the data in its own format. Everything must first go into a "dataset object" (here `StandardDataset`) that bundles the features, the label and the protected attribute together, and records which group is privileged and which is unprivileged. The metric classes we use then compare across exactly two groups, the privileged one against the unprivileged one. This two-group aspect is the detail to keep in mind: when dealing with more than 2 groups, this will lead to a comparison between 1 group and an average of the others, which will be interesting to observe in terms of what it does to our results. 

### Fairlearn

Fairlearn comes from Microsoft. It is a leaner, more focused library, which consequently also carries fewer metrics and fewer mitigation methods. It was built to feel like a natural extension of scikit-learn, so it works directly on plain arrays: the true labels, the predicted labels, and a column of sensitive features. There is no special data container to build first (Bird et al., 2020).

Its central object is the `MetricFrame`, which computes a metric separately for every group it is given and then reports the spread between them. It can work with any number of groups, not just 2, and by default it reports the worst case, the largest gap between any pair of groups. This is one of the main differences that we will look into as opposed to AIF360 (which, as previously mentioned, takes a different approach when more than 2 groups are involved). 

### Same definitions, different aggregation

The two libraries are not measuring different concepts. The three fairness ideas in this project map almost one to one:

| Fairness idea | AIF360 | Fairlearn |
|---|---|---|
| Demographic parity | `statistical_parity_difference` | `demographic_parity_difference` |
| Equalized odds | `average_odds_difference` | `equalized_odds_difference` |
| Proportional parity | `disparate_impact` (a ratio) | `demographic_parity_ratio` |

What differs is how each one brings its groups down to a single number. AIF360 reports a signed difference between the two fixed groups, so it keeps the direction of the bias (who is favoured) but only ever looks at one pair. Fairlearn reports an unsigned worst-case gap across however many groups it was given, so it drops the direction but catches the widest disparity anywhere in the data. On a binary protected attribute these two ways of summarising land on the same magnitude, which is why on our German Credit dataset, this should make the toolkits look interchangeable. On a six-group attribute they can come apart, which is what we expect with the COMPAS dataset (more on the datasets later). 

### Mitigation: different stages of the pipeline

Similarly, there are differences in how each library addresses bias once identified, and we deliberately picked one method from each so the comparison spans different points in the pipeline.

AIF360 Reweighing (Section 5) is a pre-processing method (Kamiran & Calders, 2012). It leaves the data values and the model untouched and only changes how much each training row counts, so that the favourable outcome is balanced across groups before training begins. It is simple, model-agnostic, and aimed at demographic parity. Because it works off the same privileged-versus-unprivileged split as AIF360's metrics, it only ever balances two groups.

Fairlearn ExponentiatedGradient (Section 6) is an in-processing method (Agarwal et al., 2018). Instead of touching the data, it retrains the model repeatedly under an explicit fairness constraint, trading a little accuracy for a smaller gap, and it can hold that constraint across all the groups at once. So it attacks the problem from inside the model, and it is naturally multi-group.

Neither is more 'correct' than the other, they just tackle things differently, and seeing where those differences make them disagree is the point of this project.

### The two datasets, and why the difference between them matters

As hinted at earlier, the project works with 2 datasets that are deliberately different, so that any disagreement between the toolkits can be traced to a property of the data rather than to some exceptional example.

German Credit is the smaller and cleaner of the two. We chose it as it was suggested to us in class to look at a more Europe-focused dataset, and this is the one we came across that we felt made sense for the scope of the project. It holds 1,000 loan applicants from a German bank, originally from the Statlog project and now hosted by the UCI Machine Learning Repository (Hofmann, 1994). Each applicant is described by 20 attributes, a mix of financial features (account status, credit amount, loan duration, existing credits) and personal ones (age, employment, housing). The label is a simple good or bad credit risk, split 700 good to 300 bad. The protected attribute is sex, which we derive from the applicant's personal-status code the same way AIF360 does, and it has exactly two values: male, treated here as the privileged group, and female. The favourable outcome is being judged a good credit risk.

COMPAS is larger, messier, and from a completely different domain. It comes from ProPublica's 2016 investigation into a risk-assessment tool used in Broward County, Florida (Angwin et al., 2016), and after the standard filtering we keep a little over 6000 criminal defendants. Each is described by their criminal history (prior counts, juvenile records, charge degree) and some basic demographics, and the label is whether the person was rearrested within two years, so here the favourable outcome is no recidivism. The protected attribute is race, and this is the key difference: race takes 1 of 6 values (African-American, Caucasian, Hispanic, Asian, Native American, or Other), and they are very unevenly sized; African-American and Caucasian defendants make up the bulk of the data, while Asian and Native American defendants are only a handful of rows each.

So the datasets differ in several ways:

- domain (finance versus criminal justice),
- size,
- how clean they are,
- and most importantly, the form of the protected attribute: German Credit's is binary, COMPAS's is a six-way category.

This last difference is what drives most of what follows.

### What the binary versus six-group split should lead us to expect

We mentioned that AIF360 always compares 2 fixed groups, while Fairlearn compares every group it is given and reports the worst pair. Thus, the shape of each dataset's protected attribute decides how much room that leaves for the two toolkits to disagree.

On German Credit, the protected attribute has only two groups, so both toolkits are looking at the same male vs female comparison. There is essentially nothing for them to disagree about: a difference metric computed by one should match the magnitude reported by the other, and we expect their verdicts to line up almost perfectly. Any small gaps that remain should come from wording and sign conventions, not from a real difference in what is being measured.

On COMPAS, the protected attribute has 6 groups, and the two toolkits no longer see the same picture. AIF360 collapses race into Caucasian versus everyone else and measures that one gap. Fairlearn keeps all 6 groups and reports the widest gap between any pair. We therefore expect the COMPAS numbers to drift apart, with Fairlearn generally reporting the larger disparity, since a worst-of-six gap can only be as wide as or wider than a single averaged one. We also expect the split to matter most where a small group sits at an extreme, because such a group is invisible to AIF360's binary view but can set Fairlearn's worst case on its own.

The expectation carries into mitigation. A fix that balances only the Caucasian-versus-rest aspect (which is all AIF360 Reweighing can target) should transfer cleanly on German Credit, where that aspect is the whole problem, but on COMPAS it can simply push the disparity onto a group it never balanced, so the two toolkits should agree less after mitigation than before. In summary, we expect the binary dataset to make the toolkits look interchangeable and the 6-group dataset to pull them apart, more so once we start trying to fix the bias rather than only measure it. The sections that follow will test exactly this.

### Research questions
This project asks one overarching question: to what extent do AIF360 and Fairlearn produce consistent findings when applied to algorithmic bias detection and mitigation, across different datasets and fairness metrics? Both libraries claim to measure and reduce the same kinds of bias, so in principle they should agree. We test where they actually do, and where they come apart. The question breaks into three sub-questions:

1. SQ1: Verdict and magnitude. When both toolkits evaluate the same classifier, do they agree on whether bias exists? And if they disagree, is it a difference in direction, in magnitude, or in the final biased/fair verdict?
2. SQ2: Dataset structure. Does that agreement depend on the dataset? In particular, does it change when the protected attribute is binary (sex in German Credit) versus multi-group (race in COMPAS), and does the choice of model matter too?
3. SQ3: Mitigation transfer. When we apply a fix from each toolkit (AIF360 Reweighing, a binary pre-processing method, and Fairlearn ExponentiatedGradient, a multi-group in-processing method), do the improvements transfer across toolkits, or does each fix only satisfy the toolkit it came from?

---
### AI Disclaimer

During this project, we occasionally used Claude as a learning aid while working on the coding portions of the notebook. As our programming experience is still growing, it helped us clarify syntax, unfamiliar functions, and understand how techniques such as Reweighing, the ExponentiatedGradient reduction, and the individual fairness metrics work in practice. 

Any AI support was only a starting point: we adapted the code ourseleves, annotated it, tested it, and ensured we understood it as well as possible before including it in this final project. We chose the research questions, the datasets, the models, the fairness metrics and the two mitigation methods independently. 

The interpretations and conclusions reflect our own understanding, based on knowledge from previous courses, and iterative development of the notebook throughout the course period, thus we take full responsibility for the final content.

This project took a considerable amount of time and effort, and we are very happy with what we were able to build and of how much we learned along the way.

Accordingly, we cite Claude as a reference, used in the manner described above.

---
## 0. Imports

In [1]:
# hide aif360/tensorflow warnings
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd

# classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# preprocessing and pipeline utilities
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# evaluation metrics
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report, roc_auc_score
)

# aif360: dataset container, scoring, and the Reweighing pre-processing mitigation
import aif360
from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing

# fairlearn: group metrics and the ExponentiatedGradient in-processing mitigation
from fairlearn.metrics import (
    demographic_parity_difference,   # | rate gap | across groups
    equalized_odds_difference,       # worst of (TPR gap, FPR gap)
    demographic_parity_ratio,        # min rate / max rate across groups
    MetricFrame,                     # per-group metric table
    selection_rate,                  # share predicted to the favorable outcome
)
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds

SEED = 42
print('imports ok')

pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[inFairness]'


imports ok


Everything the notebook needs is imported here in one place, so nothing further down has to pull in extra packages. pandas and numpy handle the data, scikit-learn provides the two models (logistic regression and random forest) together with the scaling, pipeline and scoring helpers, and the two toolkits we're comparing are aif360 and fairlearn, each contributing both its metrics and the one mitigation method we use from it (Reweighing from aif360, ExponentiatedGradient from fairlearn). SEED = 42 fixes the randomness so every run gives the same numbers, and the `imports ok` print is just a check that nothing failed to load.

---
## 1. Data Loading

In [2]:
# German Credit

GERMAN_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'

# column names taken from the uci documentation for this dataset
GERMAN_COLS = [
    'status', 'month', 'credit_history', 'purpose', 'credit_amount',
    'savings', 'employment', 'investment_as_income_percentage', 'personal_status',
    'other_debtors', 'residence_since', 'property', 'age',
    'installment_plans', 'housing', 'number_of_credits',
    'skill_level', 'people_liable_for', 'telephone', 'foreign_worker', 'credit'
]

# read space-separated file with no header
df_g = pd.read_csv(GERMAN_URL, sep=' ', header=None, names=GERMAN_COLS)

# derive sex from personal_status codes (same mapping as aif360's GermanDataset source)
# A91/A93/A94 = male, A92/A95 = female
df_g['sex'] = df_g['personal_status'].map(
    {'A91': 'male', 'A93': 'male', 'A94': 'male', 'A92': 'female', 'A95': 'female'}
)

# convert labels to {0, 1} before passing to StandardDataset
# the raw file uses 1=good, 2=bad — aif360 would keep them as {1,2} otherwise
df_g['credit'] = (df_g['credit'] == 1).astype(float)  # good=1.0, bad=0.0

# wrap in StandardDataset: handles one-hot encoding of categoricals
# and maps protected attributes to binary privileged/unprivileged (1/0)
german = StandardDataset(
    df=df_g,
    label_name='credit',
    favorable_classes=[1],            # 1 = good credit = favorable outcome
    protected_attribute_names=['sex'],
    privileged_classes=[['male']],    # male = privileged group
    categorical_features=[
        'status', 'credit_history', 'purpose', 'savings', 'employment',
        'personal_status', 'other_debtors', 'property', 'installment_plans',
        'housing', 'skill_level', 'telephone', 'foreign_worker'
    ],
    features_to_drop=['personal_status'],  # original column, replaced by 'sex'
)

# COMPAS

# compas data comes packaged in aif360, so no download needed
COMPAS_PATH = os.path.join(
    os.path.dirname(aif360.__file__),
    'data', 'raw', 'compas', 'compas-scores-two-years.csv'
)

# read raw csv, indexed on 'id' so we can recover race per row later
df_c = pd.read_csv(COMPAS_PATH, index_col='id', na_values=[])

# apply the same five filters that aif360's CompasDataset uses internally
df_c = df_c[
    (df_c['days_b_screening_arrest'] <= 30) &
    (df_c['days_b_screening_arrest'] >= -30) &
    (df_c['is_recid'] != -1) &
    (df_c['c_charge_degree'] != 'O') &
    (df_c['score_text'] != 'N/A')
]

# derived feature used by aif360's default compas preprocessing
df_c['length_of_stay'] = (
    pd.to_datetime(df_c['c_jail_out']) - pd.to_datetime(df_c['c_jail_in'])
).dt.days

compas = StandardDataset(
    df=df_c,
    label_name='two_year_recid',
    favorable_classes=[0],                        # 0 = no recidivism = favorable outcome
    protected_attribute_names=['sex', 'race'],
    privileged_classes=[['Female'], ['Caucasian']],
    categorical_features=['age_cat', 'c_charge_degree', 'c_charge_desc'],
    features_to_keep=['sex', 'age', 'age_cat', 'race', 'juv_fel_count',
                      'juv_misd_count', 'juv_other_count', 'priors_count',
                      'c_charge_degree', 'c_charge_desc', 'two_year_recid',
                      'length_of_stay'],
)

print(f'German Credit : {german.features.shape[0]} rows, {german.features.shape[1]} features')
print(f'  favorable_label={german.favorable_label} (good credit)')
print(f'COMPAS        : {compas.features.shape[0]} rows, {compas.features.shape[1]} features')
print(f'  favorable_label={compas.favorable_label} (no recidivism)')

German Credit : 1000 rows, 58 features
  favorable_label=1.0 (good credit)
COMPAS        : 6167 rows, 402 features
  favorable_label=0.0 (no recidivism)


Both datasets get wrapped in AIF360's StandardDataset, a shared format that also one-hot-encodes the categorical columns, which is why German Credit ends up with 58 features. The line worth checking is favorable_label: good credit = 1 for German, no re-offending = 0 for COMPAS. The five rows COMPAS drops are just AIF360's usual cleaning filters.

---
## 2. EDA

In [3]:
# German Credit

print('=== GERMAN CREDIT ===')

# extract labels and protected attributes from the aif360 object
y_g   = german.labels.ravel()
sex_g = german.protected_attributes[:, 0]  # 1=male (privileged), 0=female
fav_g = german.favorable_label             # 1.0 = good credit

print(f'N={len(y_g)} | good credit: {(y_g == fav_g).mean():.1%} | male: {sex_g.mean():.1%}')

# good credit rate broken down by sex, shows pre-model disparity in the labels
for val, name in [(1, 'Male'), (0, 'Female')]:
    mask = sex_g == val
    print(f'  [{name}] n={mask.sum()} | good credit rate: {(y_g[mask] == fav_g).mean():.1%}')

# COMPAS

print()
print('=== COMPAS ===')

y_c    = compas.labels.ravel()
race_c = compas.protected_attributes[:, 1]  # 1=Caucasian (privileged), 0=other
fav_c  = compas.favorable_label             # 0.0 = no recidivism

print(f'N={len(y_c)} | no recid: {(y_c == fav_c).mean():.1%} | Caucasian: {race_c.mean():.1%}')

# no-recidivism rate broken down by race, shows pre-model disparity in the labels
for val, name in [(1, 'Caucasian'), (0, 'Non-Caucasian')]:
    mask = race_c == val
    print(f'  [{name}] n={mask.sum()} | no recid rate: {(y_c[mask] == fav_c).mean():.1%}')

=== GERMAN CREDIT ===
N=1000 | good credit: 70.0% | male: 69.0%
  [Male] n=690 | good credit rate: 72.3%
  [Female] n=310 | good credit rate: 64.8%

=== COMPAS ===
N=6167 | no recid: 54.5% | Caucasian: 34.1%
  [Caucasian] n=2100 | no recid rate: 60.9%
  [Non-Caucasian] n=4067 | no recid rate: 51.1%


Before any modelling, here are the outcome rates per group. In German Credit, men get good credit 72.3% of the time against 64.8% for women, a 7.5-point gap. In COMPAS, Caucasian defendants have a 60.9% no-reoffend rate versus 51.1% for non-Caucasians, about ten points. These gaps live in the labels themselves, so a model trained here tends to inherit them, and the fairness scores later on will inevitably reflect this from the data rather than only the algorithm.

---
## 3. Model Training

### Why Logistic Regression and Random Forest

We run every test on two models rather than one, and the pair is chosen on purpose. The aim is a simple comparison that reflects how these problems are actually modelled, in practice and in the fairness literature, rather than some unnecessarily fancy or complex setups whose results would not generalise.

Logistic Regression is the transparent, linear baseline. We were taught to use it as a safe first choice for a baseline in the Data Science and AI Part 1 course last semester, so it was the natural place to start here. In credit scoring it is still the industry default, kept in place largely because lenders and regulators need decisions they can explain, and a linear model's coefficients are easy to read (Lessmann et al., 2015). It is also the model ProPublica used in the original COMPAS analysis, so it links the recidivism side of the project to the original study (Larson et al., 2016). It is also one of the small bunch of standard algorithms used in the main comparative study of fairness interventions, which is the closest prior work to ours (Friedler et al., 2019).

Random Forest is the flexible, non-linear counterpart (Breiman, 2001). The same, previous course presented it to us as essentially the next step up from logistic regression, what you reach for when a straight line is not enough. It picks up interactions between features that a linear model misses, it needs little tuning, and in large benchmarks it is repeatedly among the strongest off-the-shelf classifiers. A 243-dataset head-to-head found it beats Logistic Regression on a clear majority of problems (Couronné et al., 2018), and an even broader comparison of 179 classifiers ranked the random-forest family best overall (Fernández-Delgado et al., 2014). In credit scoring specifically, it sits at or near the top of the standard benchmark (Lessmann et al., 2015).

Looking at them both together is the point. They sit at different points on the scale in terms of interpretability versus flexibility, a simple model you can read against a black box that usually predicts better, which is exactly the contrast the fairness literature reaches for when it wants results that are not tied to one model family (Friedler et al., 2019). If AIF360 and Fairlearn disagree on both of these models, the disagreement is about the toolkits and not the classifier; if they only disagree on one of the two, that tells us the divergence is depends on the model too. The two-model design is what lets the project tell those cases apart. Both models are used as implemented in scikit-learn (Pedregosa et al., 2011).

In [4]:
def train_eval(aif_ds, clf, label='', target_names=None):

    # 70/30 split via aif360 so the test set stays a StandardDataset
    train, test = aif_ds.split([0.7], shuffle=True, seed=SEED)

    # scale then fit; LR is sensitive to feature scale
    pipe = make_pipeline(StandardScaler(), clf)
    pipe.fit(train.features, train.labels.ravel())

    y_pred = pipe.predict(test.features)
    # probability of the positive class; used for auc calculation
    y_prob = pipe.predict_proba(test.features)[:, 1]

    y_test = test.labels.ravel()
    print(label)
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}  |  AUC: {roc_auc_score(y_test, y_prob):.4f}')
    print(classification_report(y_test, y_pred, target_names=target_names))

    # return the test set as an aif360 object, needed for ClassificationMetric later
    return test, pipe, y_pred, y_prob

`train_eval` is the one helper reused for all four models, so the recipe is the same each time. We give it a dataset and a classifier and it splits 70/30 (same split every run), scales the features, trains the model, and prints how it did. It also hands back the test set and predictions for the fairness section to reuse. Nothing runs here yet; the next four cells call it.

In [5]:
test_lr_g, pipe_lr_g, y_pred_lr_g, y_prob_lr_g = train_eval(
    german,
    LogisticRegression(max_iter=1000, random_state=SEED),
    label='LR | German Credit',
    target_names=['Bad (0)', 'Good (1)']
)

LR | German Credit
Accuracy: 0.7500  |  AUC: 0.8044
              precision    recall  f1-score   support

     Bad (0)       0.61      0.51      0.56        92
    Good (1)       0.80      0.86      0.83       208

    accuracy                           0.75       300
   macro avg       0.70      0.68      0.69       300
weighted avg       0.74      0.75      0.74       300



Logistic Regression on German Credit. A quick note on the scores:

- accuracy is the share of predictions that are correct,
- AUC is how well the model ranks a good case above a bad one (0.5 is guessing, 1.0 is perfect),
- recall for a class is how many of its real cases the model caught.

Accuracy 0.75 and AUC 0.80 are solid, but the model treats the classes unevenly: it catches 86% of good-credit cases and only 51% of bad-credit ones. That's normal when 70% of the data is good, since the model leans toward the majority. Worth remembering, because that lean comes back in the fairness scores.

In [6]:
test_rf_g, pipe_rf_g, y_pred_rf_g, y_prob_rf_g = train_eval(
    german,
    RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1),
    label='RF | German Credit',
    target_names=['Bad (0)', 'Good (1)']
)

RF | German Credit
Accuracy: 0.7367  |  AUC: 0.8164
              precision    recall  f1-score   support

     Bad (0)       0.88      0.16      0.28        92
    Good (1)       0.73      0.99      0.84       208

    accuracy                           0.74       300
   macro avg       0.81      0.58      0.56       300
weighted avg       0.78      0.74      0.67       300



*A note on the RF parameters: 
- n_estimators=100: the number of trees voting on each prediction. More trees give a more stable result with diminishing returns; 100 is scikit-learn's default.
- max_depth=5: caps how deep each tree grows. The default is unlimited, which would overfit on a dataset this small (~700 training rows), so this is a deliberate cap to keep the trees shallow and regularised.
- random_state=SEED: fixes the forest's randomness so the run is reproducible, using the same seed as everywhere else.
- n_jobs=-1: trains the trees in parallel across all CPU cores. For speed only, no effect on the result.

Random Forest on German Credit. The forest ranks cases a little better (AUC 0.816 vs 0.804) but pushes the same lean much further: it almost never predicts bad credit (recall 0.16, just 15 of 92 cases), so it's effectively calling nearly everyone good. This matters later. A model that gives almost everyone the same outcome looks very fair, because every group ends up with near-identical approval rates, but that's a side-effect of the model barely separating cases rather than genuine fairness.

In [7]:
test_lr_c, pipe_lr_c, y_pred_lr_c, y_prob_lr_c = train_eval(
    compas,
    clf=LogisticRegression(max_iter=1000, random_state=SEED),
    label='LR | COMPAS',
    target_names=['No recid (0)', 'Recid (1)']
)

LR | COMPAS
Accuracy: 0.6710  |  AUC: 0.7110
              precision    recall  f1-score   support

No recid (0)       0.68      0.76      0.72      1023
   Recid (1)       0.65      0.56      0.60       828

    accuracy                           0.67      1851
   macro avg       0.67      0.66      0.66      1851
weighted avg       0.67      0.67      0.67      1851



Logistic Regression on COMPAS. Accuracy 0.671, AUC 0.711, weaker than German Credit but more even between the two classes. Re-offending is simply harder to predict than credit risk, since the features carry less signal. This is the baseline COMPAS model the fairness metrics will judge.

In [8]:
test_rf_c, pipe_rf_c, y_pred_rf_c, y_prob_rf_c = train_eval(
    compas,
    clf=RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1),
    label='RF | COMPAS',
    target_names=['No recid (0)', 'Recid (1)']
)

RF | COMPAS
Accuracy: 0.6629  |  AUC: 0.7205
              precision    recall  f1-score   support

No recid (0)       0.65      0.86      0.74      1023
   Recid (1)       0.70      0.42      0.53       828

    accuracy                           0.66      1851
   macro avg       0.68      0.64      0.63      1851
weighted avg       0.67      0.66      0.64      1851



Random Forest on COMPAS. Same shape as before: the forest ranks slightly better (AUC 0.721 vs 0.711) but under-predicts the minority outcome (re-offend recall 0.42). The difference from the German forest is that this one doesn't collapse to a single answer, it still predicts re-offend fairly often, so its group disparities won't disappear the way German's did.

In [9]:
# collect all four results into a single comparison table
summary = pd.DataFrame([
    {'Dataset': 'German Credit', 'Model': 'LR',
     'Accuracy': accuracy_score(test_lr_g.labels.ravel(), y_pred_lr_g),
     'AUC': roc_auc_score(test_lr_g.labels.ravel(), y_prob_lr_g)},
    {'Dataset': 'German Credit', 'Model': 'RF',
     'Accuracy': accuracy_score(test_rf_g.labels.ravel(), y_pred_rf_g),
     'AUC': roc_auc_score(test_rf_g.labels.ravel(), y_prob_rf_g)},
    {'Dataset': 'COMPAS',        'Model': 'LR',
     'Accuracy': accuracy_score(test_lr_c.labels.ravel(), y_pred_lr_c),
     'AUC': roc_auc_score(test_lr_c.labels.ravel(), y_prob_lr_c)},
    {'Dataset': 'COMPAS',        'Model': 'RF',
     'Accuracy': accuracy_score(test_rf_c.labels.ravel(), y_pred_rf_c),
     'AUC': roc_auc_score(test_rf_c.labels.ravel(), y_prob_rf_c)},
]).set_index(['Dataset', 'Model']).round(4)

display(summary)

Accuracy     AUC
Dataset       Model                  
German Credit LR       0.7500  0.8044
              RF       0.7367  0.8164
COMPAS        LR       0.6710  0.7110
              RF       0.6629  0.7205

All four models' headline scores in one table. RF beats LR on AUC on both datasets (better ranking) but loses on accuracy, because it skews so hard toward the majority class. We're not necessarily looking for the best model here though; the point is that LR and RF predict differently, so they should also give us different fairness scores. That's what we then test in Section 4.

---
## 4. Fairness Metric Comparison

This is the core of the project. For every model and dataset combination we measure the same three fairness metrics with both toolkits, then check whether they agree.

The three metrics, in plain words:

- Demographic parity: asks 'are the approval rates equal?'. Whether each group gets the favourable outcome (good credit, or predicted no-reoffend) at the same rate. It looks at rates only, not at whether the prediction is correct (Dwork et al., 2012).
- Equalized odds: asks 'are the mistakes equal?'. Whether the model catches true cases and raises false alarms at the same rate for each group. A model can have equal approval rates and still lean more on one group (Hardt et al., 2016).
- Proportional parity: asks 'does it pass the 80% legal test?'. It's based on the four-fifths rule from US employment law: if the worst-off group's approval rate is below 80% of the best-off group's, that counts as discriminatory (U.S. Equal Employment Opportunity Commission, 1978; Feldman et al., 2015). This one is a ratio, not a difference.

Each metric corresponds directly to one function within each toolkit:

| Concept | AIF360 | Fairlearn |
|---|---|---|
| Demographic parity | `statistical_parity_difference` | `demographic_parity_difference` |
| Equalized odds | `average_odds_difference` | `equalized_odds_difference` |
| Proportional parity | `disparate_impact` | `demographic_parity_ratio` |

Which group we protect: German Credit uses sex (male = privileged), binary in both toolkits. COMPAS uses race, and this is where the toolkits diverge. AIF360 is locked to a binary split (Caucasian vs everyone else), while Fairlearn takes the full six-category race variable and reports the worst gap across all groups.

Two things to keep in mind when reading the numbers:

- Direction vs magnitude: AIF360's difference scores are signed, where the sign says which group is worse off (negative means the unprivileged group is disadvantaged), while Fairlearn's are sizes only (always at least 0), so we compare AIF360's absolute value against Fairlearn's.
- What counts as favourable: the favourable outcome is the good result for the individual (being granted good credit on German, and not reoffending on COMPAS). The catch is that the two datasets code that outcome with opposite numbers. On German the favourable label is 1 (good credit = 1, bad credit = 0), but on COMPAS it is 0 (no reoffence = 0, reoffence = 1). The metrics have to be told which value counts as favourable, otherwise they would read the advantage backwards. AIF360 takes this from each dataset’s stored favorable_label, while for Fairlearn we relabel so that 1 always means favourable. With both pointed at the same event, their scores line up.

How we call it, and where the cut-offs come from. For the difference metrics we flag a gap as biased once its size passes 0.05, that is, a five percentage point difference in rates between the groups. This is a common rule of thumb in fairness work rather than a legal standard: anything smaller is treated as practically negligible, more likely noise than systematic disadvantage. For the ratio metric we borrow the four-fifths rule from US employment law (U.S. Equal Employment Opportunity Commission, 1978): if the disadvantaged group’s favourable rate is below 80% of the best group’s, that counts as evidence of unfairness. We apply it two-sided, flagging any ratio outside 0.80 to 1.25 (1.25 is simply 1 / 0.80), so the test catches a disparity whichever group ends up ahead.

In [10]:
# protected groups for AIF360, referenced by attribute name (robust to column order)
GERMAN_PRIV, GERMAN_UNPRIV = [{'sex': 1}], [{'sex': 0}]  # male vs female
COMPAS_PRIV, COMPAS_UNPRIV = [{'race': 1}], [{'race': 0}]  # Caucasian vs rest

# verdict thresholds
DIFF_THRESH = 0.05  # |difference| above this -> biased
RATIO_BAND = (0.80, 1.25)  # ratio outside this band -> biased (four-fifths rule)

METRICS = ['Demographic Parity', 'Equalized Odds', 'Proportional Parity']

def verdict(metric, value):
    """'BIASED' / 'fair' / 'n/a' for a metric value under the project thresholds."""
    if value is None or not np.isfinite(value):
        return 'n/a'
    if metric == 'Proportional Parity':                       # ratio metric
        return 'fair' if RATIO_BAND[0] <= value <= RATIO_BAND[1] else 'BIASED'
    return 'BIASED' if abs(value) > DIFF_THRESH else 'fair'   # difference metric

def agree_label(a, b):
    """Do the two toolkits reach the same verdict?"""
    if 'n/a' in (a, b):
        return 'n/a'
    return 'yes' if a == b else 'no'

def compas_race_multicat(test_ds):
    """Recover the original 6-category race for each COMPAS test row.
    instance_names are the original COMPAS 'id' values (df_c was indexed on 'id'),
    so this .loc lookup returns race aligned 1:1 with test_ds.features and y_pred."""
    ids = [int(name) for name in test_ds.instance_names]
    return df_c.loc[ids, 'race'].to_numpy()

print('fairness setup ok')

fairness setup ok


Groundwork for the fairness section. We import the three metrics from each toolkit, set who counts as privileged vs unprivileged for each dataset (male and Caucasian = privileged), and fix the bias thresholds. The three small helpers do what they say:

- `verdict` turns a number into a biased/fair label,
- `agree_label` checks whether the two toolkits match,
- `compas_race_multicat` recovers COMPAS's full six-category race for Fairlearn.

No scores yet, hence the `fairness setup ok` print.

In [11]:
def aif_fairness(test_ds, y_pred, privileged_groups, unprivileged_groups):
    classified = test_ds.copy(deepcopy=True)
    classified.labels = np.asarray(y_pred).reshape(-1, 1)  # swap true labels for predictions
    cm = ClassificationMetric(test_ds, classified,
                              privileged_groups=privileged_groups,
                              unprivileged_groups=unprivileged_groups)
    return {
        'Demographic Parity':  cm.statistical_parity_difference(),  # P(fav|unpriv) - P(fav|priv)
        'Equalized Odds':      cm.average_odds_difference(),        # mean of (TPR gap, FPR gap)
        'Proportional Parity': cm.disparate_impact(),               # P(fav|unpriv) / P(fav|priv)
    }

def fl_fairness(y_true, y_pred, sensitive_features, favorable_label):
    """Same three concepts from Fairlearn. These are MAGNITUDES / worst-case
    across groups. Relabel so 1 == favorable, matching AIF360's convention,
    so both toolkits measure the same event (matters for COMPAS, fav = 0)."""
    yt = (np.asarray(y_true) == favorable_label).astype(int)
    yp = (np.asarray(y_pred) == favorable_label).astype(int)
    return {
        'Demographic Parity':  demographic_parity_difference(yt, yp, sensitive_features=sensitive_features),
        'Equalized Odds':      equalized_odds_difference(yt, yp, sensitive_features=sensitive_features),
        'Proportional Parity': demographic_parity_ratio(yt, yp, sensitive_features=sensitive_features),
    }

These two functions do the actual scoring. `aif_fairness` runs a model's predictions through AIF360, comparing the two groups we give it; `fl_fairness` does the same through Fairlearn but scans every group it's given. Both return the same three metrics, so we can line them up directly. One small thing worth remembering from before: because COMPAS's good outcome is 0 (no re-offend), we relabel inside `fl_fairness` so both toolkits measure the same event.

---
### 4.1 Score all four model × dataset combinations

For each combo we give AIF360 the binary protected attribute and Fairlearn the group vector to scan (binary sex for German Credit, six-category race for COMPAS), then display every result into one clean table.

In [12]:
# (dataset, model, aif360 test set, predictions, favorable label,
#  aif360 priv groups, aif360 unpriv groups, fairlearn sensitive-feature vector)
combos = [
    ('German Credit', 'LR', test_lr_g, y_pred_lr_g, german.favorable_label,
        GERMAN_PRIV, GERMAN_UNPRIV,
        np.where(test_lr_g.protected_attributes[:, 0] == 1, 'male', 'female')),
    ('German Credit', 'RF', test_rf_g, y_pred_rf_g, german.favorable_label,
        GERMAN_PRIV, GERMAN_UNPRIV,
        np.where(test_rf_g.protected_attributes[:, 0] == 1, 'male', 'female')),
    ('COMPAS', 'LR', test_lr_c, y_pred_lr_c, compas.favorable_label,
        COMPAS_PRIV, COMPAS_UNPRIV, compas_race_multicat(test_lr_c)),
    ('COMPAS', 'RF', test_rf_c, y_pred_rf_c, compas.favorable_label,
        COMPAS_PRIV, COMPAS_UNPRIV, compas_race_multicat(test_rf_c)),
]

rows = []
for ds, mdl, test_ds, y_pred, fav, priv, unpriv, sens in combos:
    aif = aif_fairness(test_ds, y_pred, priv, unpriv)               # AIF360, binary
    fl  = fl_fairness(test_ds.labels.ravel(), y_pred, sens, fav)    # Fairlearn, full group scope
    for m in METRICS:
        av, fv = aif[m], fl[m]
        rows.append({
            'Dataset': ds, 'Model': mdl, 'Metric': m,
            'AIF360': av, 'AIF360_verdict': verdict(m, av),
            'Fairlearn': fv, 'Fairlearn_verdict': verdict(m, fv),
        })

fairness_results = pd.DataFrame(rows)
fairness_results['Agree'] = [
    agree_label(a, b) for a, b in
    zip(fairness_results['AIF360_verdict'], fairness_results['Fairlearn_verdict'])
]
fairness_results.round(4)

,Dataset,Model,Metric,AIF360,AIF360_verdict,Fairlearn,Fairlearn_verdict,Agree
0,German Credit,LR,Demographic Parity,-0.2253,BIASED,0.2253,BIASED,yes
1,German Credit,LR,Equalized Odds,-0.2406,BIASED,0.3283,BIASED,yes
2,German Credit,LR,Proportional Parity,0.7258,BIASED,0.7258,BIASED,yes
3,German Credit,RF,Demographic Parity,0.0131,fair,0.0131,fair,yes
4,German Credit,RF,Equalized Odds,0.0235,fair,0.0326,fair,yes
5,German Credit,RF,Proportional Parity,1.0140,fair,0.9862,fair,yes
6,COMPAS,LR,Demographic Parity,-0.1770,BIASED,0.2351,BIASED,yes
7,COMPAS,LR,Equalized Odds,-0.1494,BIASED,0.6047,BIASED,yes
8,COMPAS,LR,Proportional Parity,0.7593,BIASED,0.6802,BIASED,yes
9,COMPAS,RF,Demographic Parity,-0.2200,BIASED,0.2999,BIASED,yes


The raw scores. Each row is one metric, for one model, on one dataset, measured by both toolkits. To read a row: AIF360 is its score (signed, so a minus means the unprivileged group is worse off), Fairlearn is its score (always positive, just the size of the gap), each `_verdict` is that toolkit's biased/fair call, and `Agree` is yes when the two calls match. The next two cells break down this table further. 

---
### 4.2 Headline comparison and toolkit agreement

Now we line the two toolkits up side by side and compute an agreement score: the share of cells where AIF360 and Fairlearn reach the same biased/fair verdict. Going in, we expect COMPAS to agree less, since Fairlearn sees six race groups where AIF360 sees two, so this is where we find out whether this assumption is valid.

In [13]:
# side-by-side view: one row per (Dataset, Model, Metric)
compare = (fairness_results
           .set_index(['Dataset', 'Model', 'Metric'])
           [['AIF360', 'AIF360_verdict', 'Fairlearn', 'Fairlearn_verdict', 'Agree']]
           .round(4))
display(compare)

# agreement score per dataset (ignoring any 'n/a' cells)
scored = fairness_results[fairness_results['Agree'].isin(['yes', 'no'])]
agreement = (scored.assign(match=scored['Agree'].eq('yes'))
             .groupby('Dataset')['match'].mean()
             .mul(100).round(1)
             .rename('Toolkit agreement (%)')
             .to_frame())
display(agreement)

AIF360 AIF360_verdict  Fairlearn  \
Dataset       Model Metric                                                  
German Credit LR    Demographic Parity  -0.2253         BIASED     0.2253   
                    Equalized Odds      -0.2406         BIASED     0.3283   
                    Proportional Parity  0.7258         BIASED     0.7258   
              RF    Demographic Parity   0.0131           fair     0.0131   
                    Equalized Odds       0.0235           fair     0.0326   
                    Proportional Parity  1.0140           fair     0.9862   
COMPAS        LR    Demographic Parity  -0.1770         BIASED     0.2351   
                    Equalized Odds      -0.1494         BIASED     0.6047   
                    Proportional Parity  0.7593         BIASED     0.6802   
              RF    Demographic Parity  -0.2200         BIASED     0.2999   
                    Equalized Odds      -0.2030         BIASED     0.3142   
                    Proportional Parity  0.7498         BIASED     0.6701   

                                        Fairlearn_verdict Agree  
Dataset       Model Metric                                       
German Credit LR    Demographic Parity             BIASED   yes  
                    Equalized Odds                 BIASED   yes  
                    Proportional Parity            BIASED   yes  
              RF    Demographic Parity               fair   yes  
                    Equalized Odds                   fair   yes  
                    Proportional Parity              fair   yes  
COMPAS        LR    Demographic Parity             BIASED   yes  
                    Equalized Odds                 BIASED   yes  
                    Proportional Parity            BIASED   yes  
              RF    Demographic Parity             BIASED   yes  
                    Equalized Odds                 BIASED   yes  
                    Proportional Parity            BIASED   yes

,Toolkit agreement (%)
Dataset,
COMPAS,100.0
German Credit,100.0


The top table lines the toolkits up so you can compare them row by row, and the bottom table is the agreement score. The surprise is that it's 100% for both datasets: every biased/fair call matches. But looking closely at the raw numbers, they often differ a lot (COMPAS equalized odds: AIF360 -0.15 vs Fairlearn 0.60). So the toolkits actually agree on the verdict while disagreeing on the size, which is exactly what the next cell measures.

---
### 4.3 How far apart are the toolkits in magnitude?

The agreement score was a blunt yes/no, and it came back 100% on both datasets, the same cells flagged by both. But agreeing on the verdict isn't the same as agreeing on the number: two scores can both say 'biased' while one is twice the size of the other.

So the cell below measures the toolkit gap, meaning how far apart AIF360's and Fairlearn's scores are on each row. For every metric we line up the two numbers, ignore AIF360's sign (it only marks direction, which Fairlearn doesn't report), measure the distance between them, and average those distances per metric and dataset. A small gap means the two libraries land on nearly the same number; a large gap means they disagree on how biased the model is, even when they agree that it is.

In [14]:
# magnitude divergence between the toolkits, per metric and dataset
fr = fairness_results.copy()
is_ratio = fr['Metric'].eq('Proportional Parity')
fr['Toolkit gap'] = np.where(
    is_ratio,
    (fr['AIF360'] - fr['Fairlearn']).abs(),            # ratio: compare directly
    (fr['AIF360'].abs() - fr['Fairlearn']).abs(),      # differences: ignore AIF360's sign
)

# average the gap for each metric, per dataset -> one column per metric
divergence = (fr.groupby(['Dataset', 'Metric'])['Toolkit gap']
              .mean().round(4)
              .unstack('Metric')
              .reindex(columns=['Demographic Parity', 'Equalized Odds', 'Proportional Parity']))
display(divergence)

Metric,Demographic Parity,Equalized Odds,Proportional Parity
Dataset,,,
COMPAS,0.069,0.2832,0.0794
German Credit,0.000,0.0484,0.0139


How big the disagreement is, metric by metric. Each cell is the average gap between the two toolkits' scores for that metric on that dataset, averaged over the two models. On German Credit every gap is tiny (0.00 to 0.05), so the libraries essentially agree on the numbers. On COMPAS every gap is larger, and equalized odds is the standout at about 0.28, roughly six times the German figure. That's the binary-vs-multigroup effect: on COMPAS Fairlearn scans all six race groups and reports the worst one, while AIF360 only ever compares Caucasian against everyone else, so its numbers stay smaller. Fuller interpretation below.

---
### 4.4 What the comparison shows

The verdicts agree everywhere (100% on both datasets), but the agreement is thinner than it looks. The real story is in the magnitudes, and it splits by dataset. On German Credit, where sex is binary, the two toolkits see the same groups and barely differ. On COMPAS they pull apart, because Fairlearn scans all six race groups for the worst pair while AIF360 stays on its Caucasian-vs-rest split. The clearest case is COMPAS LR equalized odds, 0.149 from AIF360 against 0.605 from Fairlearn, a 4x gap on one model. The verdicts only match because both land on the same side of the 0.05 limit; put the threshold between those two values and the toolkits would disagree. So the 100% is more a fact about where the threshold sits, not proof that the two measure the same thing.

The verdict also depends more on which classifier you audit than on which toolkit you use. On German Credit the two models disagree completely: logistic regression is flagged biased on all three metrics, while the random forest is judged fair on all three. That looks like good news for the forest, but it is misleading. As Section 3 showed, this forest is degenerate, because it predicts good credit for almost everyone, so men and women end up with nearly the same approval rate and the gap between them shrinks to almost nothing. It scores as fair only because it has stopped telling applicants apart, not because it treats the two groups more equally. The COMPAS random forest does not behave this way, so its disparities remain. So switching the classifier can flip the verdict, while switching the toolkit does not.

---
## 5. Bias Mitigation: AIF360 Reweighing (pre-processing)

Sections 1 to 4 only measured bias. This is the first intervention: we apply AIF360's flagship mitigation, Reweighing (Kamiran & Calders, 2012), and ask whether a fix built inside one toolkit still looks like a fix under the other toolkit's metrics. That's the core of sub-question 3.

What Reweighing does: it's a pre-processing method, so it never touches the model or the feature values. Instead it gives every training row a weight so that, in the weighted data, the label becomes statistically independent of the protected attribute. Groups under-represented among the favourable outcomes get up-weighted, and the rest down-weighted. The classifier then trains on this re-weighted sample.

This is what keeps the experiment clean. The classifier, its hyper-parameters and the 70/30 split (seed=42) are identical to the baseline in Sections 1 to 3, so the only thing that changes is sample_weight, and every score shift below is down to the mitigation rather than a modelling choice.

The asymmetry is the interesting part. Reweighing is binary-locked: it balances exactly one privileged/unprivileged split, sex for German Credit and the binary race axis (Caucasian vs everyone else) for COMPAS. But when we re-score, Fairlearn still scans all six COMPAS race groups. So the open question is whether fixing the binary axis also closes the worst-case multi-group gap, or whether bias just re-surfaces in a group Reweighing never saw.

We mitigate all four baseline models (LR and RF, German and COMPAS) and re-score each with both toolkits

### 5.1 Apply Reweighing and retrain

In [15]:
def reweigh_eval(aif_ds, clf, priv, unpriv, label='', target_names=None):
    train, test = aif_ds.split([0.7], shuffle=True, seed=SEED)  # identical split to baseline

    rw = Reweighing(privileged_groups=priv, unprivileged_groups=unpriv)
    train_rw = rw.fit_transform(train)  # adds .instance_weights

    pipe = make_pipeline(StandardScaler(), clf)
    step = pipe.steps[-1][0]  # the final estimator step
    pipe.fit(train_rw.features, train_rw.labels.ravel(),
             **{f'{step}__sample_weight': train_rw.instance_weights})

    y_pred = pipe.predict(test.features)
    y_prob = pipe.predict_proba(test.features)[:, 1]
    y_test = test.labels.ravel()
    print(label)
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}  |  AUC: {roc_auc_score(y_test, y_prob):.4f}')
    print(classification_report(y_test, y_pred, target_names=target_names))
    return test, pipe, y_pred, y_prob

# Reweigh on the same binary axis the measurement layer uses:
#   German -> sex (male = privileged); COMPAS -> race (Caucasian = privileged, binary)
test_lr_g_rw, pipe_lr_g_rw, y_pred_lr_g_rw, y_prob_lr_g_rw = reweigh_eval(
    german, LogisticRegression(max_iter=1000, random_state=SEED),
    GERMAN_PRIV, GERMAN_UNPRIV, label='LR | German Credit | Reweighed',
    target_names=['Bad (0)', 'Good (1)'])

test_rf_g_rw, pipe_rf_g_rw, y_pred_rf_g_rw, y_prob_rf_g_rw = reweigh_eval(
    german, RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1),
    GERMAN_PRIV, GERMAN_UNPRIV, label='RF | German Credit | Reweighed',
    target_names=['Bad (0)', 'Good (1)'])

test_lr_c_rw, pipe_lr_c_rw, y_pred_lr_c_rw, y_prob_lr_c_rw = reweigh_eval(
    compas, LogisticRegression(max_iter=1000, random_state=SEED),
    COMPAS_PRIV, COMPAS_UNPRIV, label='LR | COMPAS | Reweighed',
    target_names=['No recid (0)', 'Recid (1)'])

test_rf_c_rw, pipe_rf_c_rw, y_pred_rf_c_rw, y_prob_rf_c_rw = reweigh_eval(
    compas, RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1),
    COMPAS_PRIV, COMPAS_UNPRIV, label='RF | COMPAS | Reweighed',
    target_names=['No recid (0)', 'Recid (1)'])

LR | German Credit | Reweighed
Accuracy: 0.7533  |  AUC: 0.8077
              precision    recall  f1-score   support

     Bad (0)       0.62      0.51      0.56        92
    Good (1)       0.80      0.86      0.83       208

    accuracy                           0.75       300
   macro avg       0.71      0.69      0.69       300
weighted avg       0.74      0.75      0.75       300

RF | German Credit | Reweighed
Accuracy: 0.7300  |  AUC: 0.8177
              precision    recall  f1-score   support

     Bad (0)       0.92      0.13      0.23        92
    Good (1)       0.72      1.00      0.84       208

    accuracy                           0.73       300
   macro avg       0.82      0.56      0.53       300
weighted avg       0.78      0.73      0.65       300

LR | COMPAS | Reweighed
Accuracy: 0.6591  |  AUC: 0.7036
              precision    recall  f1-score   support

No recid (0)       0.67      0.75      0.71      1023
   Recid (1)       0.64      0.55      0.59       82

Each model was retrained on the reweighted data and tested on the same rows as its baseline, so the accuracy and AUC here are directly comparable to Sections 1 to 3. The numbers barely move: the biggest drop is COMPAS LR, from 0.671 to 0.659, and German LR even ticks up from 0.750 to 0.753, with the AUCs staying flat. This matters because a fix that worked only by breaking the model would show up as a big accuracy loss here. Since the cost stays small, the fairness changes we see later are real, not the side effect of a broken model. The reweighed predictions (`y_pred_*_rw`) are then scored with the same code used in Section 4.

### 5.2 Re-score the mitigated models with both toolkits

The raw fairness scores for the four reweighed models, computed with the identical `aif_fairness` and `fl_fairness` functions from Section 4. COMPAS still hands Fairlearn the full six-category race variable, so the binary-vs-multigroup contrast carries over into mitigation. This table is meant to be read like the Section 4 one, just that these are the post-mitigation numbers.

In [16]:
# Reweighed combos mirror the baseline `combos`, swapping in the mitigated predictions.
combos_rw = [
    ('German Credit', 'LR', test_lr_g_rw, y_pred_lr_g_rw, german.favorable_label,
        GERMAN_PRIV, GERMAN_UNPRIV,
        np.where(test_lr_g_rw.protected_attributes[:, 0] == 1, 'male', 'female')),
    ('German Credit', 'RF', test_rf_g_rw, y_pred_rf_g_rw, german.favorable_label,
        GERMAN_PRIV, GERMAN_UNPRIV,
        np.where(test_rf_g_rw.protected_attributes[:, 0] == 1, 'male', 'female')),
    ('COMPAS', 'LR', test_lr_c_rw, y_pred_lr_c_rw, compas.favorable_label,
        COMPAS_PRIV, COMPAS_UNPRIV, compas_race_multicat(test_lr_c_rw)),
    ('COMPAS', 'RF', test_rf_c_rw, y_pred_rf_c_rw, compas.favorable_label,
        COMPAS_PRIV, COMPAS_UNPRIV, compas_race_multicat(test_rf_c_rw)),
]

def score_combos(combo_list):
    """Both toolkits over a combos list -> tidy long df (same columns as fairness_results)."""
    rows = []
    for ds, mdl, test_ds, y_pred, fav, priv, unpriv, sens in combo_list:
        aif = aif_fairness(test_ds, y_pred, priv, unpriv)             # AIF360, binary
        fl  = fl_fairness(test_ds.labels.ravel(), y_pred, sens, fav)  # Fairlearn, full group scope
        for m in METRICS:
            av, fv = aif[m], fl[m]
            rows.append({'Dataset': ds, 'Model': mdl, 'Metric': m,
                         'AIF360': av, 'AIF360_verdict': verdict(m, av),
                         'Fairlearn': fv, 'Fairlearn_verdict': verdict(m, fv)})
    out = pd.DataFrame(rows)
    out['Agree'] = [agree_label(a, b)
                    for a, b in zip(out['AIF360_verdict'], out['Fairlearn_verdict'])]
    return out

fairness_results_rw = score_combos(combos_rw)
fairness_results_rw.round(4)

,Dataset,Model,Metric,AIF360,AIF360_verdict,Fairlearn,Fairlearn_verdict,Agree
0,German Credit,LR,Demographic Parity,-0.1421,BIASED,0.1421,BIASED,yes
1,German Credit,LR,Equalized Odds,-0.1425,BIASED,0.1900,BIASED,yes
2,German Credit,LR,Proportional Parity,0.8215,fair,0.8215,fair,yes
3,German Credit,RF,Demographic Parity,0.0222,fair,0.0222,fair,yes
4,German Credit,RF,Equalized Odds,0.0397,fair,0.0722,BIASED,no
5,German Credit,RF,Proportional Parity,1.0234,fair,0.9772,fair,yes
6,COMPAS,LR,Demographic Parity,-0.0147,fair,0.2602,BIASED,no
7,COMPAS,LR,Equalized Odds,0.0178,fair,0.2948,BIASED,no
8,COMPAS,LR,Proportional Parity,0.9765,fair,0.6820,BIASED,no
9,COMPAS,RF,Demographic Parity,-0.1710,BIASED,0.2738,BIASED,yes


### 5.3 Did the fix work? Movement toward fairness and verdict agreement

Two tables answer sub-question 3, each explained directly beneath it. The accuracy cost was already covered in 5.1, and Section 7 reports it in full across every version.

In [17]:
# How far is each metric from its fairness target, before vs after reweighing?
#   difference metrics -> target 0 (compared by magnitude) ; ratio metric -> target 1
# Delta toward fair = (distance at baseline) - (distance after): >0 improved, <0 got worse.
TARGET = {'Demographic Parity': 0.0, 'Equalized Odds': 0.0, 'Proportional Parity': 1.0}

def stack(toolkit):
    b = fairness_results.set_index(['Dataset', 'Model', 'Metric'])[toolkit].rename('Baseline')
    r = fairness_results_rw.set_index(['Dataset', 'Model', 'Metric'])[toolkit].rename('Reweighed')
    out = pd.concat([b, r], axis=1).reset_index()
    out.insert(3, 'Toolkit', toolkit)
    return out

def dist_to_target(values, metrics):
    """Distance from the fairness target: |value| for differences, |value - 1| for the ratio."""
    d = values.abs()
    is_ratio = metrics.eq('Proportional Parity')
    return d.where(~is_ratio, (values - 1.0).abs())

mit = pd.concat([stack('AIF360'), stack('Fairlearn')], ignore_index=True)
mit['Dist_base']     = dist_to_target(mit['Baseline'],  mit['Metric'])
mit['Dist_rw']       = dist_to_target(mit['Reweighed'], mit['Metric'])
mit['Δ toward fair'] = mit['Dist_base'] - mit['Dist_rw']
mit['Effect'] = np.select(
    [~np.isfinite(mit['Baseline']) | ~np.isfinite(mit['Reweighed']),
     mit['Δ toward fair'] >  1e-9,
     mit['Δ toward fair'] < -1e-9],
    ['n/a', 'improved', 'WORSE'], default='—')

mit_view = (mit.set_index(['Dataset', 'Model', 'Toolkit', 'Metric'])
              [['Baseline', 'Reweighed', 'Δ toward fair', 'Effect']]
              .round(4).sort_index())
display(mit_view)

Baseline  Reweighed  \
Dataset       Model Toolkit   Metric                                     
COMPAS        LR    AIF360    Demographic Parity    -0.1770    -0.0147   
                              Equalized Odds        -0.1494     0.0178   
                              Proportional Parity    0.7593     0.9765   
                    Fairlearn Demographic Parity     0.2351     0.2602   
                              Equalized Odds         0.6047     0.2948   
                              Proportional Parity    0.6802     0.6820   
              RF    AIF360    Demographic Parity    -0.2200    -0.1710   
                              Equalized Odds        -0.2030    -0.1539   
                              Proportional Parity    0.7498     0.7994   
                    Fairlearn Demographic Parity     0.2999     0.2738   
                              Equalized Odds         0.3142     0.2570   
                              Proportional Parity    0.6701     0.6989   
German Credit LR    AIF360    Demographic Parity    -0.2253    -0.1421   
                              Equalized Odds        -0.2406    -0.1425   
                              Proportional Parity    0.7258     0.8215   
                    Fairlearn Demographic Parity     0.2253     0.1421   
                              Equalized Odds         0.3283     0.1900   
                              Proportional Parity    0.7258     0.8215   
              RF    AIF360    Demographic Parity     0.0131     0.0222   
                              Equalized Odds         0.0235     0.0397   
                              Proportional Parity    1.0140     1.0234   
                    Fairlearn Demographic Parity     0.0131     0.0222   
                              Equalized Odds         0.0326     0.0722   
                              Proportional Parity    0.9862     0.9772   

                                                   Δ toward fair    Effect  
Dataset       Model Toolkit   Metric                                        
COMPAS        LR    AIF360    Demographic Parity          0.1623  improved  
                              Equalized Odds              0.1316  improved  
                              Proportional Parity         0.2172  improved  
                    Fairlearn Demographic Parity         -0.0251     WORSE  
                              Equalized Odds              0.3098  improved  
                              Proportional Parity         0.0018  improved  
              RF    AIF360    Demographic Parity          0.0489  improved  
                              Equalized Odds              0.0491  improved  
                              Proportional Parity         0.0496  improved  
                    Fairlearn Demographic Parity          0.0261  improved  
                              Equalized Odds              0.0573  improved  
                              Proportional Parity         0.0287  improved  
German Credit LR    AIF360    Demographic Parity          0.0832  improved  
                              Equalized Odds              0.0981  improved  
                              Proportional Parity         0.0957  improved  
                    Fairlearn Demographic Parity          0.0832  improved  
                              Equalized Odds              0.1383  improved  
                              Proportional Parity         0.0957  improved  
              RF    AIF360    Demographic Parity         -0.0090     WORSE  
                              Equalized Odds             -0.0162     WORSE  
                              Proportional Parity        -0.0094     WORSE  
                    Fairlearn Demographic Parity         -0.0090     WORSE  
                              Equalized Odds             -0.0396     WORSE  
                              Proportional Parity        -0.0090     WORSE

This table tracks movement toward fairness. For each model, toolkit and metric it shows the score before and after reweighing (Baseline, Reweighed) and how much closer the metric moved to its fair target (Δ toward fair: 0 for the difference metrics, 1 for the ratio; AIF360's signed values are compared by size, to match the verdict rule). A positive Δ means the metric improved, and the Effect column flags any cell that Reweighing made worse.

One cell to watch is German Credit RF under Fairlearn: its equalized-odds gap moves the wrong way, from 0.0326 to 0.0722, which later flips its verdict from fair to biased. As 5.4 shows, this jump is just noise from a model that has stopped telling applicants apart, not a real change in fairness.

In [18]:
# does the fix change whether the two toolkits agree? (baseline = 100% on both datasets)
scored_rw = fairness_results_rw[fairness_results_rw['Agree'].isin(['yes', 'no'])]
agreement_rw = (scored_rw.assign(match=scored_rw['Agree'].eq('yes'))
                .groupby('Dataset')['match'].mean().mul(100).round(1)
                .rename('After reweighing (%)').to_frame())
agreement_rw['At baseline (%)'] = 100.0
display(agreement_rw[['At baseline (%)', 'After reweighing (%)']])

# the exact cells where the toolkits now disagree (all 12 agreed at baseline)
flipped = (fairness_results_rw.loc[fairness_results_rw['Agree'].eq('no'),
           ['Dataset', 'Model', 'Metric', 'AIF360_verdict', 'Fairlearn_verdict']]
           .reset_index(drop=True))
print(f'{len(flipped)} cells now disagree (AIF360 vs Fairlearn verdict):')
display(flipped)

,At baseline (%),After reweighing (%)
Dataset,,
COMPAS,100.0,50.0
German Credit,100.0,83.3


4 cells now disagree (AIF360 vs Fairlearn verdict):


,Dataset,Model,Metric,AIF360_verdict,Fairlearn_verdict
0,German Credit,RF,Equalized Odds,fair,BIASED
1,COMPAS,LR,Demographic Parity,fair,BIASED
2,COMPAS,LR,Equalized Odds,fair,BIASED
3,COMPAS,LR,Proportional Parity,fair,BIASED


This table asks whether the two toolkits still reach the same verdict after the fix, with the exact cells that now disagree listed underneath it. They agreed on every cell at baseline, so any disagreement here was introduced by the mitigation itself.

### 5.4 What the mitigation shows

At baseline both libraries agreed on every cell (100% on both datasets). After Reweighing, agreement drops to 83.3% on German Credit and 50% on COMPAS (see 5.3). The toolkits matched while we were only measuring bias, and they diverge once we try to fix it. That divergence is what sub-question 3 was looking for.

The cause is the binary-versus-multigroup difference. German Credit's protected attribute is binary, so both toolkits compare the same two groups and their numbers match to four decimals (German LR demographic parity improves by 0.083 under both). COMPAS does not. For COMPAS LR, AIF360 reports a clean fix: demographic parity -0.177 to -0.015, equalized odds -0.149 to +0.018, proportional parity 0.759 to 0.977, all three now fair. Fairlearn rejects the same model on all three, because scanning all six race groups still finds a wide gap. Reweighing only balanced the Caucasian-vs-rest axis it was given, so the disparity moved to a group pair it never touched. COMPAS LR demographic parity is the clearest case: Fairlearn reports it getting worse (0.235 to 0.260) while AIF360 reports it improving by 0.16. Same model, opposite verdicts.

German RF equalized odds moves the wrong way under Fairlearn (0.0326 to 0.0722), which flips the verdict from fair to biased. The doubling is most probably noise, for a few reasons. Reweighing balances selection rates (demographic parity), not the TPR and FPR that equalized odds uses, so it can move equalized odds in either direction. The forest also predicts good credit for almost everyone (bad-credit recall 0.13), so the error rates come from very few cases, mostly the small set of women with bad credit in a 300-row test set. With that few rows, one flipped prediction shifts the gap a lot. And both values are small, with the 0.05 cutoff between them, so a change that would not matter elsewhere crosses the line. A fairness score on a model that has stopped separating cases is not worth much.

Reweighing also cut some equalized-odds gaps sharply (COMPAS LR 0.60 to 0.29 under Fairlearn). That is a side effect, not a guarantee, since it does not target error rates.

---
## 6. Bias Mitigation: Fairlearn ExponentiatedGradient (in-processing)

Section 5 used AIF360 Reweighing, a pre-processing fix that only rebalances the single binary axis it is given. Here we use Fairlearn's main mitigation, ExponentiatedGradient (Agarwal et al., 2018), which works in the middle of the pipeline. It does not touch the data. Instead it retrains the classifier under an explicit fairness constraint, fitting the model again and again with adjusted weights until the constraint is met.

The key difference for our project is scope. Reweighing could only balance Caucasian against the rest on COMPAS, while ExponentiatedGradient takes the full six-group race vector and constrains the worst gap across all of them.

Three implementation choices are worth mentioning before the code:

- Constraint matching. ExponentiatedGradient optimises one constraint at a time, so we run it twice per model, once for `DemographicParity` and once for `EqualizedOdds`. We then judge each fitted model only on the metric family it was actually asked to satisfy (the demographic-parity run on demographic and proportional parity, the equalized-odds run on equalized odds). Scoring it on a constraint it never received would be unfair to it, and would not compare cleanly against Reweighing, which has no target and is scored on all three.
- Pre-scaling. ExponentiatedGradient calls `estimator.fit(X, y, sample_weight=...)` internally, and a scikit-learn `Pipeline` rejects a bare `sample_weight`. So instead of putting the scaler inside the estimator like the baseline does, we fit a `StandardScaler` on the training fold, transform both folds, and pass a plain estimator to EG. Everything else about the model stays the same as the baseline.
- Reproducibility. ExponentiatedGradient returns a randomised classifier (a mix of several fitted models), so its `predict` is not deterministic on its own. We pass `random_state=SEED` to `predict` so the run repeats. The 70/30 split and SEED are the same as Sections 3 and 5, so the test rows are exactly the baseline's and every number here is comparable to before.

### 6.1 Fit ExponentiatedGradient under each constraint

In [19]:
# Fairlearn ExponentiatedGradient: in-processing mitigation under an explicit constraint.

CONSTRAINT = {'DP': DemographicParity, 'EO': EqualizedOdds}  # one constraint per run

def eg_eval(aif_ds, make_clf, constraint_key, sens_fn, eps=0.02, label='', target_names=None):
    train, test = aif_ds.split([0.7], shuffle=True, seed=SEED)   # identical split to baseline
    scaler = StandardScaler().fit(train.features)
    Xtr, Xte = scaler.transform(train.features), scaler.transform(test.features)

    mit = ExponentiatedGradient(make_clf(), constraints=CONSTRAINT[constraint_key](), eps=eps)
    mit.fit(Xtr, train.labels.ravel(), sensitive_features=sens_fn(train))  # constrain on the group vector
    y_pred = mit.predict(Xte, random_state=SEED)                            # deterministic draw

    y_test = test.labels.ravel()
    acc, bacc = accuracy_score(y_test, y_pred), balanced_accuracy_score(y_test, y_pred)
    print(f'{label:32s} | Accuracy: {acc:.4f} | Balanced acc: {bacc:.4f}')
    return test, y_pred, acc, bacc

# sensitive-feature vectors: German constrains on binary sex, COMPAS on the full six-group race.
sex_sens  = lambda ds: np.where(ds.protected_attributes[:, 0] == 1, 'male', 'female')
race_sens = lambda ds: compas_race_multicat(ds)

# (Dataset, Model, aif360 dataset, classifier factory, EG sensitive fn,
#  aif360 priv, aif360 unpriv, favorable label, scoring sensitive fn, target names)
base_specs = [
    ('German Credit', 'LR', german,
        lambda: LogisticRegression(max_iter=1000, random_state=SEED), sex_sens,
        GERMAN_PRIV, GERMAN_UNPRIV, german.favorable_label, sex_sens, ['Bad (0)', 'Good (1)']),
    ('German Credit', 'RF', german,
        lambda: RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1), sex_sens,
        GERMAN_PRIV, GERMAN_UNPRIV, german.favorable_label, sex_sens, ['Bad (0)', 'Good (1)']),
    ('COMPAS', 'LR', compas,
        lambda: LogisticRegression(max_iter=1000, random_state=SEED), race_sens,
        COMPAS_PRIV, COMPAS_UNPRIV, compas.favorable_label, race_sens, ['No recid (0)', 'Recid (1)']),
    ('COMPAS', 'RF', compas,
        lambda: RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1), race_sens,
        COMPAS_PRIV, COMPAS_UNPRIV, compas.favorable_label, race_sens, ['No recid (0)', 'Recid (1)']),
]

# fit two EG models per baseline model (one per constraint family). RF runs are heavier.
eg_models = {}  # (Dataset, Model, constraint) -> dict of everything the scoring step needs
for ds, mdl, aif_ds, clf_f, eg_sens, priv, unpriv, fav, score_sens, tn in base_specs:
    for ck in ['DP', 'EO']:
        test_ds, y_pred, acc, bacc = eg_eval(
            aif_ds, clf_f, ck, eg_sens, label=f'{mdl} | {ds} | EG-{ck}', target_names=tn)
        eg_models[(ds, mdl, ck)] = dict(
            test=test_ds, y_pred=y_pred, acc=acc, bacc=bacc,
            priv=priv, unpriv=unpriv, fav=fav, sens=score_sens(test_ds))

LR | German Credit | EG-DP       | Accuracy: 0.7533 | Balanced acc: 0.6827
LR | German Credit | EG-EO       | Accuracy: 0.7433 | Balanced acc: 0.6694
RF | German Credit | EG-DP       | Accuracy: 0.7200 | Balanced acc: 0.5526
RF | German Credit | EG-EO       | Accuracy: 0.7500 | Balanced acc: 0.5954
LR | COMPAS | EG-DP              | Accuracy: 0.6197 | Balanced acc: 0.6095
LR | COMPAS | EG-EO              | Accuracy: 0.6310 | Balanced acc: 0.6231
RF | COMPAS | EG-DP              | Accuracy: 0.6218 | Balanced acc: 0.5894
RF | COMPAS | EG-EO              | Accuracy: 0.6218 | Balanced acc: 0.5903


This fits ExponentiatedGradient twice for each of the four baseline models, once under the demographic-parity constraint and once under equalized odds, and stores all eight fitted models in `eg_models`.

The printed lines are each model's accuracy and balanced accuracy straight from fitting. Accuracy is the share of all predictions that are correct, which the majority class can dominate on a skewed dataset; balanced accuracy averages the recall on each class instead, so it gives both classes equal weight and a model that ignores the minority class cannot hide (0.5 is the floor, equivalent to guessing). They are a first look at the cost: German Credit stays close to its baseline, while COMPAS gives up a few points. We score these models for fairness in 6.2, then come back to what the cost means in 6.3.

### 6.2 Re-score the mitigated models with both toolkits

We score each EG model with the same `aif_fairness` and `fl_fairness` functions from Section 4. Since each model was trained for a single constraint, we keep only its matched metric:

- the demographic-parity model gives the demographic-parity and proportional-parity rows (proportional parity is just the ratio version of the same selection-rate idea),
- the equalized-odds model gives the equalized-odds row.

The result is one row per (dataset, model, metric), lined up directly with the baseline and reweighed tables.

In [20]:
# Score each EG model on the metric family its constraint targeted.
METRIC_FAMILY = {'DP': ['Demographic Parity', 'Proportional Parity'], 'EO': ['Equalized Odds']}

rows = []
for (ds, mdl, ck), m in eg_models.items():
    aif = aif_fairness(m['test'], m['y_pred'], m['priv'], m['unpriv'])             # AIF360, binary axis
    fl  = fl_fairness(m['test'].labels.ravel(), m['y_pred'], m['sens'], m['fav'])  # Fairlearn, full scope
    for metric in METRIC_FAMILY[ck]:
        av, fv = aif[metric], fl[metric]
        rows.append({'Dataset': ds, 'Model': mdl, 'Constraint': ck, 'Metric': metric,
                     'AIF360': av, 'AIF360_verdict': verdict(metric, av),
                     'Fairlearn': fv, 'Fairlearn_verdict': verdict(metric, fv)})

fairness_results_eg = pd.DataFrame(rows)
fairness_results_eg['Agree'] = [agree_label(a, b) for a, b in
                                zip(fairness_results_eg['AIF360_verdict'],
                                    fairness_results_eg['Fairlearn_verdict'])]
fairness_results_eg.round(4)

,Dataset,Model,Constraint,Metric,AIF360,AIF360_verdict,Fairlearn,Fairlearn_verdict,Agree
0,German Credit,LR,DP,Demographic Parity,-0.0051,fair,0.0051,fair,yes
1,German Credit,LR,DP,Proportional Parity,0.9932,fair,0.9932,fair,yes
2,German Credit,LR,EO,Equalized Odds,-0.0766,BIASED,0.0950,BIASED,yes
3,German Credit,RF,DP,Demographic Parity,0.0126,fair,0.0126,fair,yes
4,German Credit,RF,DP,Proportional Parity,1.0132,fair,0.9869,fair,yes
5,German Credit,RF,EO,Equalized Odds,0.0231,fair,0.0391,fair,yes
6,COMPAS,LR,DP,Demographic Parity,-0.0096,fair,0.1701,BIASED,no
7,COMPAS,LR,DP,Proportional Parity,0.9844,fair,0.7732,BIASED,no
8,COMPAS,LR,EO,Equalized Odds,-0.0122,fair,0.6667,BIASED,no
9,COMPAS,RF,DP,Demographic Parity,-0.0622,BIASED,0.5000,BIASED,yes


### 6.3 What ExponentiatedGradient shows on its own scores

Before comparing the two toolkits in Section 7, we look at the EG models on their own: how much accuracy they gave up, and whether their fairness actually improved.

Accuracy cost. Adding the constraint is almost free on German Credit (accuracy stays around 0.75, the same as before) but costs about four to five points on COMPAS (LR drops from 0.67 to 0.62). The fix is cheap on the easy dataset and more expensive on the hard one.

A warning on German RF. Its balanced accuracy is only 0.55, which is barely better than guessing once both classes are counted. This is the same collapse we saw in Section 3: the model predicts good credit for almost everyone, so a fair-looking score here is not real fairness, just a model that treats everyone the same because it has stopped telling them apart.

Did the fairness improve? On German Credit, mostly yes. Demographic parity and proportional parity both move into the fair range under both toolkits. Equalized odds is the one that resists: it improves a lot but stays just over the 0.05 line, so it is still flagged. That metric is harder to satisfy because it asks the model to match two error rates at once, not just one rate.

COMPAS is where the toolkits start to disagree. After EG, AIF360 says COMPAS LR is now fair on all three metrics, while Fairlearn says it is still biased on all three. That disagreement is the main result, so we lay it out properly in Section 7.

---
## 7. Cross-Toolkit Scoring Matrix

This is the central comparison of the whole project. For each dataset and model, it puts three versions of the classifier (baseline, AIF360 Reweighed, Fairlearn ExponentiatedGradient) side by side, scored by both toolkits across the three fairness concepts. Each metric's EG value comes from the constraint that matches it (demographic and proportional parity from the demographic-parity run, equalized odds from the equalized-odds run), so the EG column is complete and comparable to the others.

For each entry in the table we report two things: the distance to the fairness target (0 for the difference metrics, 1 for the ratio) and how much that distance changed from the baseline.

When reading it, two patterns are what answer sub-question 3:

- A fix transfers when it was built for one toolkit but also improves the other toolkit's score. This suggests the two are really measuring the same thing.
- A fix causes harm when one toolkit certifies it as fixed while the other now reads the model as less fair. This is the case that matters for policy: the same model passes one audit but fails the other.

### 7.1 Baseline, Reweighed and EG side by side


In [21]:
# Three-way matrix per toolkit: Baseline | Reweighed | EG, plus distance moved toward the target.
# dist_to_target() and TARGET come from Section 5.3.
METRIC_ORDER = ['Demographic Parity', 'Equalized Odds', 'Proportional Parity']

def three_way(toolkit):
    def col(df, name):
        return df.set_index(['Dataset', 'Model', 'Metric'])[toolkit].rename(name)
    m = pd.concat([col(fairness_results, 'Baseline'),
                   col(fairness_results_rw, 'Reweighed'),
                   col(fairness_results_eg, 'EG')], axis=1).reset_index()
    m['Δ RW'] = dist_to_target(m['Baseline'], m['Metric']) - dist_to_target(m['Reweighed'], m['Metric'])
    m['Δ EG'] = dist_to_target(m['Baseline'], m['Metric']) - dist_to_target(m['EG'], m['Metric'])
    m['Metric'] = pd.Categorical(m['Metric'], METRIC_ORDER, ordered=True)
    return (m.set_index(['Dataset', 'Model', 'Metric'])
              [['Baseline', 'Reweighed', 'EG', 'Δ RW', 'Δ EG']].sort_index().round(4))

for tk in ['AIF360', 'Fairlearn']:
    print(f'===== {tk}  (Δ > 0 = moved toward fair) =====')
    display(three_way(tk))

===== AIF360  (Δ > 0 = moved toward fair) =====


Baseline  Reweighed      EG    Δ RW  \
Dataset       Model Metric                                                     
COMPAS        LR    Demographic Parity    -0.1770    -0.0147 -0.0096  0.1623   
                    Equalized Odds        -0.1494     0.0178 -0.0122  0.1316   
                    Proportional Parity    0.7593     0.9765  0.9844  0.2172   
              RF    Demographic Parity    -0.2200    -0.1710 -0.0622  0.0489   
                    Equalized Odds        -0.2030    -0.1539 -0.0628  0.0491   
                    Proportional Parity    0.7498     0.7994  0.9277  0.0496   
German Credit LR    Demographic Parity    -0.2253    -0.1421 -0.0051  0.0832   
                    Equalized Odds        -0.2406    -0.1425 -0.0766  0.0981   
                    Proportional Parity    0.7258     0.8215  0.9932  0.0957   
              RF    Demographic Parity     0.0131     0.0222  0.0126 -0.0090   
                    Equalized Odds         0.0235     0.0397  0.0231 -0.0162   
                    Proportional Parity    1.0140     1.0234  1.0132 -0.0094   

                                           Δ EG  
Dataset       Model Metric                       
COMPAS        LR    Demographic Parity   0.1674  
                    Equalized Odds       0.1373  
                    Proportional Parity  0.2251  
              RF    Demographic Parity   0.1578  
                    Equalized Odds       0.1403  
                    Proportional Parity  0.1779  
German Credit LR    Demographic Parity   0.2202  
                    Equalized Odds       0.1640  
                    Proportional Parity  0.2675  
              RF    Demographic Parity   0.0006  
                    Equalized Odds       0.0003  
                    Proportional Parity  0.0008

===== Fairlearn  (Δ > 0 = moved toward fair) =====


Baseline  Reweighed      EG    Δ RW  \
Dataset       Model Metric                                                     
COMPAS        LR    Demographic Parity     0.2351     0.2602  0.1701 -0.0251   
                    Equalized Odds         0.6047     0.2948  0.6667  0.3098   
                    Proportional Parity    0.6802     0.6820  0.7732  0.0018   
              RF    Demographic Parity     0.2999     0.2738  0.5000  0.0261   
                    Equalized Odds         0.3142     0.2570  0.5000  0.0573   
                    Proportional Parity    0.6701     0.6989  0.5000  0.0287   
German Credit LR    Demographic Parity     0.2253     0.1421  0.0051  0.0832   
                    Equalized Odds         0.3283     0.1900  0.0950  0.1383   
                    Proportional Parity    0.7258     0.8215  0.9932  0.0957   
              RF    Demographic Parity     0.0131     0.0222  0.0126 -0.0090   
                    Equalized Odds         0.0326     0.0722  0.0391 -0.0396   
                    Proportional Parity    0.9862     0.9772  0.9869 -0.0090   

                                           Δ EG  
Dataset       Model Metric                       
COMPAS        LR    Demographic Parity   0.0650  
                    Equalized Odds      -0.0620  
                    Proportional Parity  0.0931  
              RF    Demographic Parity  -0.2001  
                    Equalized Odds      -0.1858  
                    Proportional Parity -0.1701  
German Credit LR    Demographic Parity   0.2202  
                    Equalized Odds       0.2333  
                    Proportional Parity  0.2675  
              RF    Demographic Parity   0.0006  
                    Equalized Odds      -0.0065  
                    Proportional Parity  0.0008

Each toolkit gets its own block. Inside a block, the three columns are the same model scored at baseline, after Reweighing, and after EG, and the Δ columns show how far each version moved toward the fairness target (positive means fairer). Reading a row left to right shows whether a fix actually helped that metric; comparing the AIF360 block against the Fairlearn block shows whether the two toolkits agree about it. The full read is in 7.4.

### 7.2 Does the verdict still agree?


In [22]:
# Verdict agreement between the toolkits, across all three model versions, per dataset.
def agree_pct(df):
    s = df[df['Agree'].isin(['yes', 'no'])]
    return s.assign(match=s['Agree'].eq('yes')).groupby('Dataset')['match'].mean().mul(100).round(1)

agreement_all = pd.concat([
    agree_pct(fairness_results).rename('Baseline'),
    agree_pct(fairness_results_rw).rename('Reweighed'),
    agree_pct(fairness_results_eg).rename('EG'),
], axis=1)
print('Toolkit verdict agreement (%) by mitigation stage:')
display(agreement_all)

# Where do AIF360 and Fairlearn now disagree after EG? (all 12 cells agreed at baseline)
flipped_eg = (fairness_results_eg.loc[fairness_results_eg['Agree'].eq('no'),
              ['Dataset', 'Model', 'Metric', 'AIF360_verdict', 'Fairlearn_verdict']]
              .reset_index(drop=True))
print(f'{len(flipped_eg)} EG cells where the toolkits disagree:')
display(flipped_eg)

Toolkit verdict agreement (%) by mitigation stage:


,Baseline,Reweighed,EG
Dataset,,,
COMPAS,100.0,50.0,33.3
German Credit,100.0,83.3,100.0


4 EG cells where the toolkits disagree:


,Dataset,Model,Metric,AIF360_verdict,Fairlearn_verdict
0,COMPAS,LR,Demographic Parity,fair,BIASED
1,COMPAS,LR,Proportional Parity,fair,BIASED
2,COMPAS,LR,Equalized Odds,fair,BIASED
3,COMPAS,RF,Proportional Parity,fair,BIASED


The top table is the headline number: the share of metrics where the two toolkits give the same fair/biased verdict, at baseline and after each fix. It started at 100% on both datasets, so any drop is caused by the mitigation. The list below names the exact metrics where they now disagree.

### 7.3 Utility cost: accuracy of every version


In [23]:
# Utility cost: accuracy of every version, so no fairness gain is read without its price.
# AUC is reported for baseline and Reweighed only; EG returns a randomised classifier with no calibrated scores, so balanced accuracy is the honest utility gauge for it.
def acc_of(test_ds, y_pred):
    return accuracy_score(test_ds.labels.ravel(), y_pred)

base_pred = {('German Credit', 'LR'): (test_lr_g, y_pred_lr_g), ('German Credit', 'RF'): (test_rf_g, y_pred_rf_g),
             ('COMPAS', 'LR'): (test_lr_c, y_pred_lr_c),        ('COMPAS', 'RF'): (test_rf_c, y_pred_rf_c)}
rw_pred   = {('German Credit', 'LR'): (test_lr_g_rw, y_pred_lr_g_rw), ('German Credit', 'RF'): (test_rf_g_rw, y_pred_rf_g_rw),
             ('COMPAS', 'LR'): (test_lr_c_rw, y_pred_lr_c_rw),        ('COMPAS', 'RF'): (test_rf_c_rw, y_pred_rf_c_rw)}

util_rows = []
for key in base_pred:
    ds, mdl = key
    util_rows.append({
        'Dataset': ds, 'Model': mdl,
        'Acc base':  acc_of(*base_pred[key]),
        'Acc RW':    acc_of(*rw_pred[key]),
        'Acc EG-DP': eg_models[(ds, mdl, 'DP')]['acc'],
        'Acc EG-EO': eg_models[(ds, mdl, 'EO')]['acc'],
        'BalAcc EG-DP': eg_models[(ds, mdl, 'DP')]['bacc'],
        'BalAcc EG-EO': eg_models[(ds, mdl, 'EO')]['bacc'],
    })
utility_all = pd.DataFrame(util_rows).set_index(['Dataset', 'Model']).round(4)
display(utility_all)

Acc base  Acc RW  Acc EG-DP  Acc EG-EO  BalAcc EG-DP  \
Dataset       Model                                                         
German Credit LR       0.7500  0.7533     0.7533     0.7433        0.6827   
              RF       0.7367  0.7300     0.7200     0.7500        0.5526   
COMPAS        LR       0.6710  0.6591     0.6197     0.6310        0.6095   
              RF       0.6629  0.6596     0.6218     0.6218        0.5894   

                     BalAcc EG-EO  
Dataset       Model                
German Credit LR           0.6694  
              RF           0.5954  
COMPAS        LR           0.6231  
              RF           0.5903

This keeps the fairness gains honest by putting the accuracy of every version side by side. AUC is shown for baseline and Reweighed only; EG returns a randomised classifier with no probability scores, so balanced accuracy is used for it instead. The check is simply that no fairness improvement came from a model that just got worse at predicting.

### 7.4 What the comparison shows

We read the three tables in order.

The scoring matrix (7.1) shows how far each fix moved each metric, under each toolkit. The clear pattern is the gap between the two datasets. On German Credit, both fixes move the metrics toward fair by almost the same amount under AIF360 and Fairlearn, so the two toolkits tell the same story. On COMPAS they do not: under AIF360 the fixes look like they work, while under Fairlearn the same models barely improve, and on some metrics get worse.

The agreement table (7.2) turns that into one number per stage. Both datasets start at 100%. On German Credit, Reweighing drops agreement to 83.3% but EG brings it back to 100%, so the in-processing fix leaves the two toolkits agreeing again. On COMPAS it goes the other way: Reweighing drops agreement to 50%, and EG drops it further to 33.3%, the lowest in the whole study. The list of disagreements underneath is all the same shape: AIF360 calls the COMPAS model fair while Fairlearn calls it biased. So whichever fix we apply, the two toolkits end up disagreeing on COMPAS, and a model one of them certifies the other rejects.

The cost table (7.3) checks that none of this came from simply breaking the models. German Credit pays almost nothing for either fix (accuracy stays around 0.75). COMPAS pays more for EG, about four to five accuracy points, against roughly one for Reweighing. The one number to treat with care is German RF: its balanced accuracy under EG is only about 0.55, so, as we saw earlier, it predicts good credit for nearly everyone and its fair-looking scores are not worth much.

Putting the three together: the binary dataset (German Credit) keeps the toolkits agreeing once EG is applied, while the multi-group dataset (COMPAS) makes them disagree more the harder we try to fix it. Section 8 is an optional deeper look that opens up COMPAS to show exactly why.

---
## 8. COMPAS Multi-Group Decomposition (extra)

This section is extra. The three questions we set out to answer were already settled by Section 7: the toolkits disagree on COMPAS, that disagreement comes from AIF360 seeing two race groups where Fairlearn sees six, and it survives both mitigations. We did not need anything more to answer them. We include this section anyway because it is worth looking under the hood once, to see the mechanism rather than just the result. It is the part of the project where we went a step deeper than the original plan.

Here is the difference from the earlier sections. Up to now we worked only with the single fairness number each toolkit reports per metric. Those numbers told us that the two libraries disagree on COMPAS, but not exactly why, because the disagreement is hidden inside how each library turns six race groups into one number. This section opens that up: for COMPAS, we print the selection rate (the share of a group predicted to the favourable outcome, no recidivism) for each of the six groups separately, for the baseline, Reweighed, and EG models. Seeing all six side by side shows what that single number was hiding.

What connects these tables to the earlier sections is the demographic parity metric. Selection rate is exactly what demographic parity is built from, so these tables are that one metric pulled apart, group by group. The gap row at the bottom (highest minus lowest) is the same demographic parity number Fairlearn reported in Sections 4 to 7: for COMPAS LR it reads 0.2351 at baseline, 0.2602 after Reweighing and 0.1701 after EG, matching the Fairlearn column there. AIF360's demographic parity is those same rates collapsed the other way, Caucasian against the pooled rest, which is why its number comes out smaller and can give the opposite verdict. So this is where those demographic parity numbers actually come from, shown in full instead of summarised. (Equalized odds is a separate story, since it depends on error rates rather than selection rates.)

One more thing to watch is the leakage idea from Section 5. Reweighing only balanced Caucasian against the rest, so if it closes that gap but a different group drifts out, the disparity was moved rather than removed. EG was constrained on all six groups, so the question for it is whether its row of rates is genuinely flat.

So this is a deeper look rather than a new finding: it confirms and explains the Section 7 disagreement by showing, group by group, why Fairlearn keeps flagging models that AIF360 signs off.

### 8.1 Selection rate by race, all six groups


In [24]:
# Per-race selection rate on COMPAS: baseline vs Reweighed vs EG, all six groups.

FAV_C = compas.favorable_label  # 0 = no recidivism = favorable

def sel_by_race(test_ds, y_pred):
    """Selection rate (share predicted to the favorable outcome) per race group."""
    race = compas_race_multicat(test_ds)
    yp = (np.asarray(y_pred) == FAV_C).astype(int)
    mf = MetricFrame(metrics=selection_rate, y_true=yp, y_pred=yp, sensitive_features=race)
    sr = mf.by_group; sr.index.name = 'race'
    return sr

def decomposition(model):
    """One table per model: selection rate by race for the three versions + the max-min gap."""
    if model == 'LR':
        base, rw = (test_lr_c, y_pred_lr_c), (test_lr_c_rw, y_pred_lr_c_rw)
    else:
        base, rw = (test_rf_c, y_pred_rf_c), (test_rf_c_rw, y_pred_rf_c_rw)
    eg = eg_models[('COMPAS', model, 'DP')]  # DP constraint -> selection-rate target
    tab = pd.DataFrame({
        'Baseline':  sel_by_race(*base),
        'Reweighed': sel_by_race(*rw),
        'EG':        sel_by_race(eg['test'], eg['y_pred']),
    })
    tab.loc['— gap (max-min) —'] = tab.max() - tab.min()
    return tab.round(4)

for model in ['LR', 'RF']:
    print(f'===== COMPAS {model}: selection rate by race (favorable = no recidivism) =====')
    display(decomposition(model))

===== COMPAS LR: selection rate by race (favorable = no recidivism) =====


,Baseline,Reweighed,EG
race,,,
African-American,0.5141,0.5580,0.5799
Asian,0.7273,0.8182,0.6364
Caucasian,0.7351,0.6225,0.6142
Hispanic,0.7048,0.7651,0.7048
Native American,0.5000,0.7500,0.7500
Other,0.7064,0.7798,0.6606
— gap (max-min) —,0.2351,0.2602,0.1701


===== COMPAS RF: selection rate by race (favorable = no recidivism) =====


,Baseline,Reweighed,EG
race,,,
African-American,0.6092,0.6353,0.7764
Asian,0.9091,0.9091,1.0000
Caucasian,0.8791,0.8526,0.8593
Hispanic,0.8072,0.8193,0.8494
Native American,0.7500,0.7500,0.5000
Other,0.8440,0.8532,0.8899
— gap (max-min) —,0.2999,0.2738,0.5000


### 8.2 What the decomposition shows

Start with the LR table. At baseline, Caucasian defendants are predicted no recidivism most often and African-American defendants least, a gap of about 0.24 across the six groups. The three versions then behave very differently:

- Reweighing makes the gap slightly worse (0.24 to 0.26). It does pull the Caucasian and African-American rates almost level, the one axis it was told to balance, but the Asian group drifts up at the same time, so the widest gap just moves to a pair the method never touched. The disparity was relocated, not removed.
- EG is the only version that flattens the whole table, bringing the gap down to about 0.17. It is the only fix that improves all six groups at once.

The RF table is the opposite warning. EG again helps the groups that matter, lifting African-American almost level with Caucasian. But its overall gap looks the worst of the three (0.50), and that is entirely down to two of the smallest groups: Asian (every defendant predicted no recidivism) against Native American (only half). Those groups are a handful of rows each, so Fairlearn's worst-case number is being driven by noise, and EG's real improvement is hidden behind it. This is why EG looked like it went backwards on COMPAS RF in Section 7.

Together the two tables answer why the toolkits disagree on COMPAS:

- AIF360 lumps every non-Caucasian group into one before it measures anything, so the per-group gaps shown here are invisible to it, and it signs off both fixes.
- Fairlearn keeps all six groups and reports the widest gap. This is the more honest measurement, but it can be swayed by groups too small to be reliable.

The practical lesson is to report the full per-group table, not just one summary number, and to flag groups that are too small to score reliably so they do not quietly decide the verdict.

---
## 9. Conclusion

The overarching question was whether AIF360 and Fairlearn produce consistent findings, and where they diverge, whether the divergence comes from the dataset, the classifier, or the mitigation method. We can now answer the three sub-questions directly.

SQ1 (verdict and magnitude): when both toolkits score the same classifier, they agree on the verdict but not on the size of the bias. At baseline they return the same fair/biased call on every metric on both datasets (100% agreement), so for a simple yes/no audit they look interchangeable. Where they differ is magnitude: the gap between their actual numbers is much larger on COMPAS than on German Credit, because Fairlearn reports the worst gap across all six race groups while AIF360 only compares Caucasian against the rest. So at baseline the disagreement is one of magnitude, not final verdict, but that verdict agreement is fragile, it only holds because the numbers happen to fall on the same side of the threshold.

SQ2 (dataset structure): yes, agreement depends on the dataset, and the structure of the protected attribute is what drives it. On German Credit, where the attribute is binary, both toolkits see the same two groups and stay close. On COMPAS, where race has six groups, they pull apart, because the worst-of-six gap Fairlearn reports is invisible to AIF360's two-group split. The model matters too, sometimes more than the toolkit: the German verdict flips from biased under LR to fair under RF, though that fair result is an artefact of a degenerate forest (balanced accuracy about 0.55) rather than a real improvement.

SQ3 (mitigation transfer): fixes transfer on the binary dataset but not on the multi-group one. On German Credit a fix made for one toolkit also satisfies the other, and EG even restores agreement to 100%. On COMPAS each fix mostly satisfies only the toolkit it came from: AIF360 Reweighing is cleared by AIF360 but still rejected by Fairlearn (agreement drops to 50%), and Fairlearn EG looks fully fixed to AIF360 while Fairlearn still flags it (agreement drops to 33.3%, the lowest in the study). So whichever toolkit's fix you apply, the other refuses to certify it on COMPAS: the same model passes one audit and fails the other. The practical takeaway is that a fix cannot be reported without saying which groups it was optimised over, and checking it under both toolkits.

The surprising part is that the multi-group method (EG) lowers agreement on the multi-group dataset instead of restoring it. Section 8 shows this is not a failure: EG actually produces the flattest per-race rates of any model, and it is Reweighing, not EG, that widens the six-group gap by pushing the disparity onto a group it never balanced. The leftover disagreement comes from AIF360 averaging all non-Caucasian groups into one while Fairlearn keeps the worst pair, where very small groups can dominate. So worst-case multi-group metrics are the right default, as long as groups too small to be reliable are flagged rather than left to decide the verdict.

This is also why every fairness number in the notebook is reported with its accuracy cost beside it: a fair-looking score from a model that has stopped telling cases apart, like the German RF, means little on its own.

Taken together, the project's main lesson is that AIF360 and Fairlearn cannot be treated as interchangeable. They agree at the surface, on the simple biased-or-not call, but underneath they encode different assumptions about what counts as a group and what counts as the worst case, and those assumptions decide the answer as soon as the data has more than two groups or the model changes. So the choice of toolkit is itself a methodological decision: it can change the size of a reported disparity, and on a multi-group dataset it can even flip whether a mitigated model is certified as fair. A fairness result is only meaningful alongside the toolkit, the metric definition, the group set, and the model it came from.

---
## References

Agarwal, A., Beygelzimer, A., Dudík, M., Langford, J., & Wallach, H. (2018). A reductions approach to fair classification. In Proceedings of the 35th International Conference on Machine Learning (Vol. 80, pp. 60-69). PMLR.

Angwin, J., Larson, J., Mattu, S., & Kirchner, L. (2016, May 23). Machine bias. ProPublica. https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing

Bellamy, R. K. E., Dey, K., Hind, M., Hoffman, S. C., Houde, S., Kannan, K., Lohia, P., Martino, J., Mehta, S., Mojsilović, A., Nagar, S., Ramamurthy, K. N., Richards, J., Saha, D., Sattigeri, P., Singh, M., Varshney, K. R., & Zhang, Y. (2019). AI Fairness 360: An extensible toolkit for detecting and mitigating algorithmic bias. IBM Journal of Research and Development, 63(4/5), 4:1-4:15. https://doi.org/10.1147/JRD.2019.2942287

Bird, S., Dudík, M., Edgar, R., Horn, B., Lutz, R., Milan, V., Sameki, M., Wallach, H., & Walker, K. (2020). Fairlearn: A toolkit for assessing and improving fairness in AI (Tech. Rep. No. MSR-TR-2020-32). Microsoft.

Breiman, L. (2001). Random forests. Machine Learning, 45(1), 5-32. https://doi.org/10.1023/A:1010933404324

Couronné, R., Probst, P., & Boulesteix, A.-L. (2018). Random forest versus logistic regression: A large-scale benchmark experiment. BMC Bioinformatics, 19, Article 270. https://doi.org/10.1186/s12859-018-2264-5

Dwork, C., Hardt, M., Pitassi, T., Reingold, O., & Zemel, R. (2012). Fairness through awareness. In Proceedings of the 3rd Innovations in Theoretical Computer Science Conference (pp. 214-226). Association for Computing Machinery. https://doi.org/10.1145/2090236.2090255

Feldman, M., Friedler, S. A., Moeller, J., Scheidegger, C., & Venkatasubramanian, S. (2015). Certifying and removing disparate impact. In Proceedings of the 21th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining (pp. 259-268). Association for Computing Machinery. https://doi.org/10.1145/2783258.2783311

Fernández-Delgado, M., Cernadas, E., Barro, S., & Amorim, D. (2014). Do we need hundreds of classifiers to solve real world classification problems? Journal of Machine Learning Research, 15(1), 3133-3181.

Friedler, S. A., Scheidegger, C., Venkatasubramanian, S., Choudhary, S., Hamilton, E. P., & Roth, D. (2019). A comparative study of fairness-enhancing interventions in machine learning. In Proceedings of the Conference on Fairness, Accountability, and Transparency (pp. 329-338). Association for Computing Machinery. https://doi.org/10.1145/3287560.3287589

Hardt, M., Price, E., & Srebro, N. (2016). Equality of opportunity in supervised learning. In Advances in Neural Information Processing Systems (Vol. 29, pp. 3315-3323). Curran Associates.

Hofmann, H. (1994). Statlog (German Credit Data) [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C5NC77

Kamiran, F., & Calders, T. (2012). Data preprocessing techniques for classification without discrimination. Knowledge and Information Systems, 33(1), 1-33. https://doi.org/10.1007/s10115-011-0463-8

Larson, J., Mattu, S., Kirchner, L., & Angwin, J. (2016, May 23). How we analyzed the COMPAS recidivism algorithm. ProPublica. https://www.propublica.org/article/how-we-analyzed-the-compas-recidivism-algorithm

Lessmann, S., Baesens, B., Seow, H.-V., & Thomas, L. C. (2015). Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research. European Journal of Operational Research, 247(1), 124-136. https://doi.org/10.1016/j.ejor.2015.05.030

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. Journal of Machine Learning Research, 12, 2825-2830.

U.S. Equal Employment Opportunity Commission. (1978). Uniform guidelines on employee selection procedures, 29 C.F.R. § 1607.